# STR-1 — Watermarking & late data

**Break → Detect → Fix → Prove.** An event-time aggregation (per-minute counts) must tolerate
**out-of-order** events but can't wait forever. A **watermark** is the engine's promise — *"I won't
accept events older than `max(event_time) − delay`"* — that lets it **finalize** a window, **drop**
its state, and **discard** late events that fall behind the frontier.

**Pre-requisite:** the unified Spark server **and Kafka** are up (`make up`). Producers use the host
listener `localhost:29092`; Spark consumes via `kafka:9092` (`SPARK_BOOTSTRAP`). **Open the Spark UI
at http://localhost:4040** (→ *Structured Streaming*) and **kafka-ui at http://localhost:8080**.

**Laptop-safe:** every stream uses `.trigger(availableNow=True)` — Spark drains all available data
and **stops on its own** (no infinite stream). Bounded produce; checkpoint under
`.tmp/checkpoint_str1`; sink to an Iceberg table. Teardown deletes the topic and drops the sink;
`make clean` clears `.tmp/`.

See the companion writeup in [`README.md`](./README.md) and the
[troubleshooting sheet](../../docs/troubleshooting.md).

In [ ]:
from common.spark_session import spark, display_df
from common.kafka_helpers import ensure_topic, produce_events, delete_topic, SPARK_BOOTSTRAP
from pyspark.sql import functions as F
from pyspark.sql.types import StructType, StructField, StringType, LongType
import datetime, shutil

TOPIC      = "str1_clicks"
SINK       = "iceberg_catalog.default.str1_window_counts"
CHECKPOINT = ".tmp/checkpoint_str1"
WATERMARK  = "2 minutes"   # how late an event may arrive before its window is finalized
WINDOW     = "1 minute"    # event-time bucket size
BASE       = "2026-06-24 10:"   # fixed base date so windows are reproducible

# value JSON is {"id", "page", "event_time": "YYYY-MM-DD HH:MM:SS"}
schema = StructType([
    StructField("id", LongType(), True),
    StructField("page", StringType(), True),
    StructField("event_time", StringType(), True),
])
print("Spark reads via", SPARK_BOOTSTRAP, "| Spark UI: http://localhost:4040 | kafka-ui: http://localhost:8080")

## Step 0/1 — fresh topic + sink, then produce an **on-time** batch

Start clean (new topic, dropped sink, fresh checkpoint), then publish 60 events spanning `10:00`–
`10:02` (20 per minute), roughly in order. Each carries its own **`event_time`**. Two stragglers are
held back for step 5 — first establish an in-order baseline. Nothing is collected; bounded JSON is
just published to Kafka (inspect in kafka-ui).

In [ ]:
ensure_topic(TOPIC, num_partitions=1, recreate=True)
spark.sql(f"DROP TABLE IF EXISTS {SINK}")
shutil.rmtree(CHECKPOINT, ignore_errors=True)   # fresh checkpoint -> reprocess from earliest

def ontime(i):
    minute, second = i // 20, (i % 20) * 3       # minutes 0,1,2 ; seconds 0..57
    return {"id": i, "page": f"/p{i % 5}", "event_time": f"{BASE}{minute:02d}:{second:02d}"}

n = produce_events(TOPIC, 60, value_fn=ontime)
print(f"clean topic + sink; produced {n} on-time events spanning {BASE}00:00 .. {BASE}02:57")

## 2. Detect it — event-time vs processing-time & the watermark frontier

A **batch** read lets us inspect what we just published: parse the JSON, cast `event_time` to a real
`timestamp`, and compute the **frontier** = `max(event_time)` and **cutoff** = `frontier − watermark`.
Any event older than the cutoff is "too late" and a watermark will drop it. `event_time` (in the
data) is unrelated to *now* (processing-time) — that gap is exactly why late data exists.

In [ ]:
def parsed_batch():
    raw = (spark.read.format("kafka")
           .option("kafka.bootstrap.servers", SPARK_BOOTSTRAP)
           .option("subscribe", TOPIC).option("startingOffsets", "earliest").load())
    return (raw.select(F.from_json(F.col("value").cast("string"), schema).alias("e")).select("e.*")
               .withColumn("event_time", F.col("event_time").cast("timestamp")))

frontier = parsed_batch().agg(F.max("event_time").alias("f")).collect()[0]["f"]
cutoff   = frontier - datetime.timedelta(minutes=2)
print("watermark frontier (max event_time):", frontier)
print("cutoff (frontier - 2 min)         :", cutoff, "  <- events older than this are 'too late'")
print("processing-time (now)             :", datetime.datetime.now(), "  <- unrelated to event_time")

## 3. Break it — a windowed count with **no watermark** (unbounded state)

A streaming `groupBy(window(event_time, "1 minute")).count()` **without** `withWatermark(...)` has no
upper bound on lateness, so the engine keeps **every** window's state forever in case a future batch
contains an old event — on a 24/7 stream that state (and the checkpoint) grows without bound. Here we
show the result *shape* on a batch read; the real cost is the retained state, not this number.

In [ ]:
no_wm = (parsed_batch().groupBy(F.window("event_time", WINDOW)).count()
         .select(F.col("window.start").alias("win_start"), "count").orderBy("win_start"))
print("windowed counts (no watermark) — in a real stream every window stays open forever:")
no_wm.show(truncate=False)

## 4. Fix it — `withWatermark` before the windowed agg, written to Iceberg

Run it as a **real stream** with `.withWatermark("event_time", "2 minutes")`, a 1-minute window, and
**`outputMode("append")`** — a watermarked windowed agg emits each window **once**, when the watermark
closes it (the natural fit for appending finalized windows to Iceberg). `.trigger(availableNow=True)`
drains the topic and **stops on its own**; `awaitTermination()` returns when the batch is done. Run #1
processes the on-time batch, then we read the sink.

A window appears in the sink only once the watermark moves past it. The **latest** minute may still be
**open** (its end is within `watermark` of the frontier), so it won't appear yet — that's the watermark
holding it for possible late arrivals.

In [ ]:
def run_windowed_stream():
    raw = (spark.readStream.format("kafka")
           .option("kafka.bootstrap.servers", SPARK_BOOTSTRAP)
           .option("subscribe", TOPIC).option("startingOffsets", "earliest").load())
    events = (raw.select(F.from_json(F.col("value").cast("string"), schema).alias("e")).select("e.*")
                 .withColumn("event_time", F.col("event_time").cast("timestamp")))
    windowed = (events.withWatermark("event_time", WATERMARK)
                .groupBy(F.window("event_time", WINDOW)).count())
    q = (windowed.writeStream.format("iceberg").outputMode("append")
         .option("checkpointLocation", CHECKPOINT)
         .trigger(availableNow=True).toTable(SINK))   # drain available data, then STOP
    q.awaitTermination()
    return q.lastProgress

def dropped(prog):
    return (prog or {}).get("stateOperators", [{}])[0].get("numRowsDroppedByWatermark", "n/a")

def show_sink(msg):
    print(msg)
    (spark.table(SINK).select(F.col("window.start").alias("win_start"),
                              F.col("window.end").alias("win_end"), "count")
          .orderBy("win_start").show(truncate=False))

prog1 = run_windowed_stream()      # RUN #1 — on-time batch
print("run #1 dropped-by-watermark:", dropped(prog1))
show_sink("sink after run #1 (on-time batch):")

## 5. Break-then-prove — a **late** event for an already-finalized window

Publish one straggler stamped `10:00:30` — a click for the `10:00` window the watermark has **already
finalized**. Its `event_time` is far behind `frontier − 2 min`, so re-running the stream
(`availableNow`, same checkpoint → resumes and processes only the new event) must **drop** it: the
`10:00` count must **not** change, and `numRowsDroppedByWatermark` ticks up.

In [ ]:
produce_events(TOPIC, 1, value_fn=lambda i: {"id": 9000, "page": "/p0", "event_time": f"{BASE}00:30"})
print(f"produced 1 LATE event for window {BASE}00 (well behind the watermark frontier)")

prog2 = run_windowed_stream()      # RUN #2 — just the late straggler
print("run #2 dropped-by-watermark:", dropped(prog2), "  <- the late event was discarded")
show_sink("sink after run #2 (late event arrived):  10:00 count is UNCHANGED")

## 6. Prove it — accepted vs dropped, with/without the late data

Two numbers move in opposite directions: the **accepted** count for the finalized `10:00` window does
**not** increase, while **dropped-by-watermark** ticks up by 1. For contrast, a plain **batch** count
that includes the late event (no watermark) *would* show `10:00` one higher — exactly what the
watermark protected the closed window from.

In [ ]:
contrast = (parsed_batch().groupBy(F.window("event_time", WINDOW)).count()
            .select(F.col("window.start").alias("win_start"),
                    F.col("count").alias("count_incl_late")).orderBy("win_start"))
watermarked = (spark.table(SINK)
               .select(F.col("window.start").alias("win_start"),
                       F.col("count").alias("count_watermarked")))
print("Side-by-side: watermarked sink (late dropped) vs no-watermark batch (late included)")
contrast.join(watermarked, "win_start", "left").orderBy("win_start").show(truncate=False)
print("=> the 10:00 window differs by exactly the 1 late event the watermark dropped.")

## Takeaways & "in real production…"

- **Event-time, not processing-time.** Bucket by the timestamp in the data; treat arrival time as
  unreliable. The watermark reconciles the two.
- **Tune the watermark to your real lateness SLA** — a direct trade-off: **too tight** drops valid
  late data (silent under-counting); **too loose** grows state/checkpoints and delays results. Set the
  delay above your measured p99 lateness.
- **Watch `numRowsDroppedByWatermark`** (Spark UI → Structured Streaming). Rising drops = events
  arriving later than the watermark allows — widen it or fix the upstream delay.
- **Watermark + `append` + idempotent sink (Iceberg).** Finalized windows write once and late
  stragglers can't corrupt closed windows. The window state lives in the **checkpoint** — the subject
  of **STR-2** (idempotency, checkpoints & restart).

## Teardown

Delete the Kafka topic and drop the Iceberg sink. `make clean` also clears `.tmp/` (the
`checkpoint_str1` checkpoint, warehouses, event logs).

In [ ]:
delete_topic(TOPIC)
spark.sql(f"DROP TABLE IF EXISTS {SINK}")
shutil.rmtree(CHECKPOINT, ignore_errors=True)
print("Deleted topic, dropped sink, removed checkpoint. `make clean` clears any remaining .tmp/ state.")